# Biopython {#sec-biopython}

*Somebody already wrote the classes you need for sequences, sequence files, alignments, and protein structures. This chapter shows you how to find your way around them.*

In [ ]:
import sandbox_widget

Now that you are familiar with Python modules (@sec-modules-and-packages) and classes (@sec-clases), you are ready to reap the rewards of other peoples work. There are tons of useful modules you can install as Python packages. Most of them are made by individual contributors, but the larger ones, like Biopython, are developed and maintained by communities of volunteers. Biopython is big, and this chapter only shows you the parts you are most likely to need. Everything here is described in much more detail in the [Biopython Tutorial](https://biopython.org/docs/latest/Tutorial/).

## Sequences {#sec-biopython-seq}

The class you will use most is `Seq`. Calling `Seq("ATGGCC")` runs its `__init__` and hands you a `Seq` object, just as `Point(4, 3)` handed you a `Point`. If you peeked behind the curtain, you would see methods relevant to sequences, like `reverse_complement` and `translate`, but also the same special methods you wrote yourself, like `__str__` and `__len__`. You already have Biopython installed. So you can import the `Seq` class from the Seq module inside the big Bio module. Many modules can have nested modules inside them (which are just sub-folders and *.py files inside other folders). To import the `Seq` class, you ask to import it from the `Bio.Seq` sub-module:

In [ ]:
from Bio.Seq import Seq

 Whenever a Biopython object puzzles you, you can always see which type it is using `type()`. You an also see the documentation of an class or function, using the help `help` built-in function, Try this:

In [ ]:
help(Seq)

You make a `Seq` object by calling the class, passing a DNA or protein string to as argumnet:

In [ ]:
dna = Seq("ATGGCCATTGTAATGGGCCGCTGA")

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Everything you know about strings works on a `Seq`. Decide what each line prints before you run the cell.

In [ ]:
%%sandbox
from Bio.Seq import Seq

dna = Seq("ATGGCCATTGTAATGGGCCGCTGA")
print(dna)
print(len(dna))
print("GCC" in dna)


Which special methods are called to produce each of the values printed?

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

You did not add a `__getitem__` method to your `Point` class but `Seq` has one. This is what makes indexing and slicing work so that a `Seq` object behaves almost like a string. Pay special attention to the last line: when you slice a `Seq`, what type of object do you get back?

In [ ]:
print(dna[:3])
print(dna[-3:])
print(dna[:3] == "ATG")
print(type(dna[:3]))

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

A `Seq` is like a string in one more way. Remember from @sec-immutable_strings that you cannot change a string in place? Predict what happens when you try with a `Seq`, then run it and read the last line of the error message.

In [ ]:
%%sandbox
from Bio.Seq import Seq

dna = Seq("ATGGCC")
dna[0] = "C"

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Predict what each line prints before you run it. You probably cannot guess what the `to_stop=True` argument does, so that one you need to look up in Biopython's [online documentation](https://biopython.org/docs/latest/).

In [ ]:
dna = Seq("ATGTGATAA")
print(dna.complement())
print(dna.reverse_complement())
print(dna.transcribe())
print(dna.translate())
print(dna.translate(to_stop=True))

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Decide what each of the two lines prints. Again, the [online documentation](https://biopython.org/docs/latest/) is your friend. Then run the cell to check.

In [ ]:
orf = Seq("ATGTGATAA")
print(orf.translate())
print(orf.translate(table=2))

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

In the translation project you wrote your own `translate_orf` function, and the project's tests defined its correct behavior. Here are three of the test inputs from the project tests: '`'ATGCCCATGTGA'`', `'ATGATNATGTGA'`, `'ATGTGA'`. Does `Seq('ATGCCCATGTGA').translate()` return the same as your `translate_orf()'ATGCCCATGTGA')`? How about with `'ATGATNATGTGA'`, and `'ATGTGA'`? If they do not, is that because one of them is wrong or is it because their documentation (and tests) specify different behaviors?

In [ ]:
for orf in ["ATGCCCATGTGA", "ATGATNATGTGA", "ATGTGA"]:
    print(orf, Seq(orf).translate())

## Sequence records {#sec-biopython-seqrecord}

A `SeqRecord` is Biopython's grown-up version of the simple `Gene` class you wrote in @sec-classes.

In [ ]:
from Bio.SeqRecord import SeqRecord

record = SeqRecord(dna, id="N42", name="orf1", description="A made-up ORF")
print(record)

It has an `id`, a `name` and a `description` attribute. And it keeps a `Seq` object as its `seq` attribute. You access the attributes with the `.` operator (like `p.x` for the x-value for a point):

In [ ]:
print(record.id)
print(record.description)
print(record.seq)

## Reading and writing sequence files {#sec-biopython-seqio}

In the projects you read sequences from files by opening the files and reading one line at a time in a for loop. That works if each line contains one sequence. However, the most common sequence formats split each sequence record over multiple lines. The Fasta format is the simples one. Have a look at this [Fasta record](https://raw.githubusercontent.com/munch-group/instructing-machines/main/docs/python/data/NM_148900.4.fasta){target="_blank"} for the human FOXP2 gene. The first line begins with a ">" character and contains only a sequence id and an optional description. The sequence itself is split across lines below it.  If you had to, I am sure you could produce code that would read and write Fasta records, but Biopython's `SeqIO` module will do that for you and it reads and writes many other file formats too. Import it like this:

In [ ]:
from Bio import SeqIO

You will rarely make a `SeqRecord` object yourself unless you produce some sequence data you want to use with Biopython. Lets pretend you have found three small ORFs in your project on finding open reading frames and want to write them to a file. You make a `SeqRecord` object for each one and put them in a list like this:

In [ ]:
my_records = [
    SeqRecord(Seq("ATGGCCATTGTAATGGGCCGCTGA"), id="orf1", description="first"),
    SeqRecord(Seq("ATGAAACCCGGGTTTTAA"), id="orf2", description="second"),
    SeqRecord(Seq("ATGTGATAA"), id="orf3", description="third"),
]

Now you can pass the list to the `write` function defined in the `SeqIO` module along with the name of the file to write to and the name of the format you want, in this case "fasta":

In [ ]:
SeqIO.write(my_records, "my_orfs.fasta", "fasta")

`SeqIO.write` returned `3`, the number of records it wrote to the file. You can see in the sidebar of VS Code that `my_orfs.fasta` was created. Open it in VS Code and have a look. 

Now lets pretend you come back later and want to read it back into Python. You do that using `SeqIO`s `parse` function, which returns an object you can *iterate* over in a for loop.

In [ ]:
for rec in SeqIO.parse("my_orfs.fasta", "fasta"):
    print(rec.id, len(rec), rec.seq.translate(to_stop=True))

`SeqIO.parse` reads one record at a time as the for-loop asks for it, just like a file object serves one line at a time (@sec-read_from_file). If it read in all the records before handing each one to the for loop, a file with millions of records could fill up you computer's memory. This way, you only keep one record in memory at a time. If you *do* want all the records at once, just then the value returned by `SeqIO.parse` into a list:

In [ ]:
list(SeqIO.parse("my_orfs.fasta", "fasta"))

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

`SeqIO` has a sibling of `parse` called `read`, which you use when the file holds exactly one record. It then returns that record directly, so you do not need a loop. What do you think happens if you use `read` on a file with three records? Decide before you run it.

In [ ]:
%%sandbox
from Bio import SeqIO

rec = SeqIO.read("my_orfs.fasta", "fasta")
print(rec.id)

## GenBank files {#sec-biopython-genbank}

A FASTA file holds little more than names and sequences. A GenBank record holds a sequence together with all the associated information from the database. In the data folder of your instructing-machines folder, the `NM_148900.4.gb` file contains a GenBank record for the same FOXP2 sequence you looked at as a Fasta record above. Before you go on , have a look at it [here](https://raw.githubusercontent.com/munch-group/instructing-machines/main/docs/python/data/NM_148900.4.gb){target="_blank"} or open it from the data folder in VS code. I am sure would would rather not have to parse that file yourself. To read a file with GenBank records, you just give `SeqIO.parse` `"genbank"` instead of `"fasta"` as its last argument: 

In [ ]:
for rec in SeqIO.parse("data/NM_148900.4.gb", "genbank"):
    print(rec.id)

For files like this one with only a single record in it, there is not point to the for loop. In that case, use the `read` function instead, which reads in that single record:

In [ ]:
record = SeqIO.read("data/NM_148900.4.gb", "genbank")

To write the GenBank record to a file in Fasta format, you just use `"fasta"` as the third argument to `SeqIO.write()`:

In [ ]:
SeqIO.write(record, "NM_148900.4.fasta", "fasta")

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

The GenBank record has the same `id`, `name`, and `description` attributes as the Fasta records. The rest of the header of the file ends up in the `annotations` attribute, which is an ordinary dictionary.  I am sure you remember the dictionary methods from @sec-dictionaries, so you can easily list the keys this dictionary. What are they? Inspect the value associated with each one. Are they all strings, and if not, what types are they, and how do you dig into what is in them?

### Sequence features {#sec-biopython-features}

In the [FOXP2 GenBank file](https://raw.githubusercontent.com/munch-group/instructing-machines/main/docs/python/data/NM_148900.4.gb){target="_blank"}, scroll down to the `FEATURES` section. It lists the exon, intron, CDS, and other regions of the sequence. Each one is read into a `SeqFeature` object and put in a list you can access as the `features` of the resulting sequence record.

Let us extract the coding sequence. Each `SeqFeature` object has the the following attributes: `extract`, `id`, `location`, `qualifiers`, `translate`, `type`. The `type` attribute says if the feature is a CDS, exon, intron, or something else. The code below loops over the features to extract a list of only the `CDS` features (in this case a single one):

In [ ]:
cds_features = []
for feature in record.features:
    if feature.type == "CDS":
        cds_features.append(feature)

cds = cds_features[0] # extract the single CDS feature

Knowing the start and end indices, we could slice out the CDS, but you would need to check if it is on the negative strand in which case you would also need to reverse-complement the resulting sequence:

In [ ]:
cds_seq = record.seq[cds.location.start:cds.location.end]
if cds.location.strand == '-':
    cds_seq = cds_seq.reverse_complement()
cds_seq

To make this common task easy, the sequence features have an `extract` method that extracts the feature's sequence:

In [ ]:
cds_seq = cds.extract(record.seq)
cds_seq

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Each `SeqFeature` has a `qualifiers` attribute that holds a dictionary with additional information about the feature. For the CDS, one is the official translation. See if you can fish it out and then compare to it to what `cds_seq.translate(to_stop=True)` returns.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

See if you can splice out the introns. It basically amounts to extracting and concatenating all features except the introns. Hint: `Seq` objects has a `join` method like strings so you can do: `Seq('').join([Seq('ATG'), Seq('TAG')])` to produce `Seq('ATGTAG')`.

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Run the cell below:

In [ ]:
record = SeqIO.read("data/NM_148900.4.gb", "genbank")
for feature in record.features:
    print(feature.type, feature.location)

The code print says the CDS is at `[556:2755](+)` (the `(+)` just says the feature is on the forward strand). Now open the GenBank file and find the line where the `CDS` starts. As you can see, the file says the CDS is at `557..2755`, Is one of them wrong or are there more than one approach to enumerating positions?

## Fetching records from NCBI {#sec-biopython-entrez}

If you wonder where that GenBank file came from, it was downloaded from NCBI's nucleotide database, which you can search in the browser at [ncbi.nlm.nih.gov](https://www.ncbi.nlm.nih.gov) along with many other databases.

Sometimes you want to automate that kind of data retrieval using Python code in a python script or notebook. The module `Bio.Entrez` lets your program do the same searches and downloads as you can do using a browser. 

NCBI asks that you send at most three requests per second (Biopython makes sure of that for you), and if you need to send more than a hundred requests in a row, you should do it on a weekend or outside US office hours. You should provide your email address, so they can contact you if your program misbehaves, rather than just blocking requests from your computer. 

The cell below downloads the record we have been using. Put your own email address in it before you run it.

In [ ]:
from Bio import Entrez

Entrez.email = "your@post.au.dk"   # use your own address

handle = Entrez.efetch(db="nucleotide",  # database to search
                       id="NM_148900.4", # id of sequence to download
                       rettype="gb",     # format of downloaded record
                       retmode="text") 

# read from the handle as you would from a file
seq_record = SeqIO.read(handle, "genbank")

# write the record to a file on your harddisk
SeqIO.write(seq_record, "NM_148900.4.gb", "genbank")

If you do not know the accession number (here NM_148900.4), you can search database enntries with `Entrez.esearch`. The search term uses the same syntax as the search box on the NCBI website, and `Entrez.read` turns NCBI's answer into a dictionary:

In [ ]:
handle = Entrez.esearch(
    db="nucleotide", 
    term="FOXP2[Gene] AND Homo sapiens[Organism] AND srcdb_refseq[prop]", 
    retmax=10)

result = Entrez.read(handle)

Then you use the get the records found to see what they are:

In [ ]:
for seq_id in result["IdList"]:
    handle = Entrez.efetch(db="nucleotide", id=seq_id, rettype="gb", retmode="text") 
    record = SeqIO.read(handle, "genbank")
    print(record.id, record.description)

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Change the search so it finds the FOXP2 sequences of the chimpanzee (*Pan troglodytes*) instead. Then fetch one of them with `Entrez.efetch`, read it with `SeqIO.read`, and print its description and length.

#### Exercise

[AI: Explainer](../intro/course-introduction.qmd#sec-badge-explainer){.small}

The search term above contains `srcdb_refseq[prop]`. Ask the assistant what that part of the term means and what would change if you left it out. Then settle it by running the search with and without it and comparing the two values of `result["Count"]`.

## Pairwise alignment {#sec-biopython-align}

We will revisit the top of alignment later in the course, but we include a bit on alignment here because it is also part of Biopython. You make a `PairwiseAligner` object and then use its `align` method to align two sequences:

In [ ]:
from Bio import Align

aligner = Align.PairwiseAligner()
alignments = aligner.align("GAACT", "GAT")

print(alignments.score)
print(len(alignments))
for alignment in alignments:
    print(alignment)

There are two different alignments with the same, best score, and the aligner gives you both. The numbers in the margins are positions in each sequence, counted Python's way from 0.

The score comes from the aligner's scoring settings, which are attributes on the aligner object. By default a match scores 1, a mismatch scores 0, and every gap position scores -1:

In [ ]:
print(aligner.match_score)
print(aligner.mismatch_score)
print(aligner.open_gap_score)
print(aligner.extend_gap_score)

#### Exercise

[SOLO](../intro/course-introduction.qmd#sec-badge-solo){.small}

Use the four scores above to compute the score of each of the two alignments by hand, position by position, the way you would do substitution and reduction steps: replace each column by its score, then add them up. Do you get the score the aligner printed?

Change the attributes and you change the alignment. Setting `mode` to `"local"` asks for the best-scoring part of the two sequences rather than the whole of both. For proteins, you usually score with a substitution matrix like BLOSUM62, which gives similar amino acids a positive score even if they are not identical:

In [ ]:
from Bio.Align import substitution_matrices

protein_aligner = Align.PairwiseAligner()
protein_aligner.mode = "local"
protein_aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
protein_aligner.open_gap_score = -10
protein_aligner.extend_gap_score = -0.5

best = protein_aligner.align("HEAGAWGHEE", "PAWHEAE")[0]
print(best.score)
print(best)

#### Exercise

[AI: Explainer](../intro/course-introduction.qmd#sec-badge-explainer){.small}

Look at the line in the middle of a printed alignment, the one made of `|`, `.` and `-`. Ask the assistant what each of the three symbols means. Then check its answer against one of the alignments you printed yourself, column by column. In the protein alignment, is every `.` really a mismatch between two different amino acids?